In [0]:
# %sql
# CREATE TABLE gizmobox.bronze.refunds (  
#     refund_id INT PRIMARY KEY,  
#     payment_id INT NOT NULL,  
#     refund_timestamp TIMESTAMP NOT NULL,  
#     refund_amount DECIMAL(10, 2) NOT NULL,  
#     refund_reason VARCHAR(255) NOT NULL 
# );

In [0]:
# %sql
# INSERT INTO  gizmobox.bronze.refunds (refund_id, payment_id, refund_timestamp, refund_amount, refund_reason)  
# VALUES  
# (1, 66, '2025-01-10 11:30:00', 85.75, 'Payment Error:Retailer'),  
# (2, 69, '2025-01-03 12:40:15', 120.50, 'Order Cancelled:Customer'),  
# (3, 72, '2025-01-06 14:45:30', 65.00, 'Product Returned:Customer'),  
# (4, 73, '2025-01-07 16:10:45', 210.99, 'Order Cancelled:Customer'),  
# (5, 75, '2025-01-09 18:25:00', 45.20, 'Payment Error:Retailer'),  
# (6, 80, '2025-01-10 09:35:20', 130.15, 'Order Cancelled:Customer'),  
# (7, 83, '2025-01-12 11:20:40', 150.00, 'Product Returned:Customer'),  
# (8, 85, '2025-01-14 13:15:30', 89.99, 'Order Cancelled:Customer'),  
# (9, 89, '2025-01-15 15:00:00', 78.50, 'Payment Error:Retailer'),  
# (10, 91, '2025-01-17 16:45:15', 250.75, 'Product Returned:Customer');

In [0]:
%sql
select * from gizmobox.bronze.refunds

In [0]:
df = spark.sql("select * from gizmobox.bronze.refunds")
from pyspark.sql.functions import split

df = df.withColumn("source_of_refund", split(df["refund_reason"], ":")[0]) \
       .withColumn("reason_for_refund", split(df["refund_reason"], ":")[1])

display(df)

In [0]:
%sql
select *,split(refund_reason,":")[0] as source_of_refund, split(refund_reason,":")[1] as reason_for_refund from gizmobox.bronze.refunds

In [0]:
%sql
select 
    refund_id,
    payment_id,
    cast(date_format(refund_timestamp, 'yyyy-MM-dd') as date)as refund_date,
    cast(date_format(refund_timestamp, 'HH:mm:ss') as timestamp) as refund_time,
    refund_timestamp,
    refund_amount,
    refund_reason,
    regexp_extract(refund_reason, '^([^:]+):', 1) as reason_for_refund,
    regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) as source_of_refund  
from gizmobox.bronze.refunds

In [0]:
%sql
create or replace table gizmobox.silver.refunds as 
select 
    refund_id,
    payment_id,
    cast(date_format(refund_timestamp, 'yyyy-MM-dd') as date)as refund_date,
    cast(date_format(refund_timestamp, 'HH:mm:ss') as timestamp) as refund_time,
    refund_timestamp,
    refund_amount,
    refund_reason,
    regexp_extract(refund_reason, '^([^:]+):', 1) as reason_for_refund,
    regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) as source_of_refund  
from gizmobox.bronze.refunds